STEP 1: Install Required Libraries

In [1]:
!pip install langchain langchain-community langchain-core
!pip install chromadb pypdf sentence-transformers
!pip install langgraph openai
!pip install -U langchain langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-community sentence-transformers
!pip install -U langchain-huggingface
!pip install -U chromadb langchain-community
!pip install -U langchain-openai
!pip install -U langchain-community transformers

STEP 2: Load Your PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("C:/Users/srini/Downloads/Customer_Support.pdf")  # 👈 put your PDF name here
documents = loader.load()

print(len(documents))

2


STEP 3: Chunk the Document

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

4


STEP 4: Create Embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

STEP 5: Store in ChromaDB

In [5]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

STEP 6: Create Retriever

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

STEP 7: LLM Setup

In [7]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=200
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
C:\Users\srini\AppData\Local\Temp\ipykernel_39960\3773143874.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


STEP 8: RAG Function

In [8]:
def rag_pipeline(query):
    docs = retriever.invoke(query)

    if not docs:
        return "Sorry, I couldn't find the answer."

    text = docs[0].page_content

    lines = text.split("\n")

    for i, line in enumerate(lines):
        if query.lower() in line.lower():
            # return next line (answer)
            if i + 1 < len(lines):
                return lines[i + 1].strip()

    return "Answer not found clearly."

STEP 9: LangGraph Workflow

In [9]:
from langgraph.graph import StateGraph

# Define state
class State(dict):
    query: str
    answer: str
    confidence: str

# Node 1: Process Query
def process_node(state):
    query = state.get("query", "")   # ✅ avoids crash
    
    answer = rag_pipeline(query)
    
    confidence = check_confidence(str(answer))
    
    return {
        "query": query,              # ✅ keep state flowing
        "answer": answer,
        "confidence": confidence
    }
# Node 2: Output
def output_node(state):
    print("\nFinal Answer:\n")
    print(state.get("answer", "No answer"))
    return state# Build Graph
builder = StateGraph(State)

builder.add_node("process", process_node)
builder.add_node("output", output_node)

builder.set_entry_point("process")
builder.add_edge("process", "output")

graph = builder.compile()

STEP 10: Conditional Routing (HITL)

In [10]:
def check_confidence(answer):
    if "I don't know" in answer or len(answer) < 20:
        return "low"
    return "high"

def process_node(state):
    query = state.get("query", "")   # ✅ avoids crash
    
    answer = rag_pipeline(query)
    
    confidence = check_confidence(str(answer))
    
    return {
        "query": query,              # ✅ keep state flowing
        "answer": answer,
        "confidence": confidence
    }
def human_node(state):
    print("\n⚠️ Escalated to Human Support")
    human_response = input("Enter human response: ")
    
    return {
        "query": state.get("query", ""),
        "answer": human_response
    }
builder = StateGraph(State)
builder.add_node("process", process_node)
builder.add_node("output", output_node)
builder.add_node("human", human_node)

builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    lambda state: state["confidence"],
    {
        "high": "output",
        "low": "human"
    }
)

builder.add_edge("human", "output")

graph = builder.compile()


STEP 10: Run the Chatbot

In [11]:
while True:
    query = input("\nAsk your question (type 'exit' to quit): ")
    
    if query == "exit":
        break
    
    graph.invoke(State({"query": query}))


Ask your question (type 'exit' to quit):  What is the return policy?



Final Answer:

Customers can return products within 7 days if damaged or defective.



Ask your question (type 'exit' to quit):  exit
